# 你的第一个实验室（Day 1）

### 请先读完本节

即使很长，也值得读：它交代课程怎么练、笔记本怎么跑、以及和后续「多 Agent 商业方案」的关系。

也请阅读 [README.md](../README.md)！更多更新视频与资源见：[课程资源页](https://edwarddonner.com/2024/11/13/llm-engineering-resources/)

## 你的第一个「前沿大模型」小项目

课程结束时，你会做出由多个 Agent 协作解决业务问题的方案。现在我们从更小的事开始：

**目标：** 写一种新型「网页阅读器」——给它一个 URL，它返回摘要（Internet Reader's Digest）。

开始前，请完成 README 里链接的环境搭建。

### 如果这是你第一次用 Notebook（Jupyter / 实验室）

点击下方每个**代码单元格**，按 **Shift+Enter** 运行。务必从上到下按顺序执行。

入门指南在 [Guides 文件夹](../guides/01_intro.ipynb)。

## 需要帮助时

有问题请在平台留言，或发邮件到 ed@edwarddonner.com，或 LinkedIn：https://www.linkedin.com/in/eddonner/  
也在尝试 X：[@edwarddonner](https://x.com/edwarddonner)

## 更多排错

见 [troubleshooting](../setup/troubleshooting.ipynb)。末尾有诊断脚本。

## 如果这些你已经会了

可以快速过前几周实验；后面会加深，最终微调自己的模型去对比商业 API。

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">请阅读 — 重要说明</h2>
            <span style="color:#900;">我的授课方式可能和其他课不同：不会在你盯着屏幕时现场狂敲代码，而是像这样跑 Jupyter，让你建立直觉。建议你<strong>看完讲座之后</strong>自己仔细跑一遍：加 print、改变体。有 GitHub 的话，用它展示你的变体——既是练习，也能给未来客户/雇主看。</span>
        </td>
    </tr>
</table>
<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">此代码是实时资源 — 请留意邮件/公告</h2>
            <span style="color:#f71;">我会定期推送更新：更好的注释、新示例、新模型（如 DeepSeek）。视频里有的这里都有，但文字版可能更新。Udemy 左侧「公告」可看更新；也可在通知设置里接收邮件。</span>
        </td>
    </tr>
</table>
<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">这些练习的商业价值</h2>
            <span style="color:#181;">笔记本既要好学，也尽量有趣（模型讲笑话、互相抬杠）。但核心是可落地的商业技能：摘要、抽取、工作流。学每个技术时，想想怎么用到你的业务里。</span>
        </td>
    </tr>
</table>


### 如需安装 Cursor / VS Code 扩展

1. 菜单 **View → Extensions**
2. 搜索 **Python**，安装 Microsoft 的 `ms-python`
3. 搜索 **Jupyter**，安装 `ms-toolsai` 的 Jupyter

### 接着选择内核（Kernel）

点右上角 **Select Kernel** → **Python Environments...** → 选带 `.venv` 且标记推荐的那项（例如 `.venv/bin/python`）。

出问题？去 troubleshooting 笔记本。

**注意：** 每个笔记本都要各自选一次内核。


In [1]:
# ========== 导入依赖：环境变量 / 抓取工具 / 展示 / OpenAI ==========

# 标准库 os：读环境变量
import os
# load_dotenv：把 .env 密钥加载进进程，避免写死在代码里
from dotenv import load_dotenv
# 本目录 scraper 工具：给 URL，返回网页正文文本
from scraper import fetch_website_contents
# 笔记本里渲染 Markdown
from IPython.display import Markdown, display
# OpenAI 官方 SDK 客户端
from openai import OpenAI

# 若本格报错，请去 troubleshooting 笔记本（英文提示保留）
# If you get an error running this cell, then please head over to the troubleshooting notebook!


# 连接到 OpenAI（或 Ollama）

下一格会从 `.env` 加载环境变量，并检查 `OPENAI_API_KEY`。

若想用免费 **Ollama**，见 README「付费 API 的免费替代」；完整示例也在 solutions 的 `day1_with_ollama.ipynb`。

## 排错提示

- **NameError**：是否从上到下跑过所有单元格？见 Python 基础指南。
- 仍不行：打开 [troubleshooting](../setup/troubleshooting.ipynb)。
- 或联系 ed@edwarddonner.com。

成本顾虑：README 有说明；也可用 Ollama（第 2 天会讲）。


In [2]:
# ========== 加载 .env，并粗查 OPENAI_API_KEY ==========

# override=True：用 .env 覆盖进程里已有的同名环境变量
load_dotenv(override=True)
# 取出密钥字符串，做形态检查
api_key = os.getenv('OPENAI_API_KEY')

# Check the key — 下列 print 文案保持英文（排错指引）
if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


# 先快速调用一次 Frontier 模型，当作「连通性预览」


In [3]:
# ========== 预览：构造最简单的 messages（仅 user） ==========
# 调用 OpenAI 就是「拼 messages → create」。有问题去 troubleshooting。

# 发给模型的用户话术（英文原文保留）
message = "Hello, GPT! This is my first ever message to you! Hi!"

# OpenAI Chat Completions 期望的列表：这里只有一条 user
messages = [{"role": "user", "content": message}]

# 笔记本会展示该列表，便于你看清结构
messages


[{'role': 'user',
  'content': 'Hello, GPT! This is my first ever message to you! Hi!'}]

In [4]:
# ========== 真正发请求：gpt-5-nano + 上一格的 messages ==========

# 默认客户端：自动读环境变量 OPENAI_API_KEY
openai = OpenAI()

# chat.completions.create：非流式；model id 字符串保持原样
response = openai.chat.completions.create(model="gpt-5-nano", messages=messages)
# 取第一条 choice 里 assistant 的文本内容
response.choices[0].message.content


'Hi there! Welcome—the pleasure is mine to help you.\n\nI’m here to help with a lot of things, from explanations and writing to planning and problem solving. A few quick ideas of what I can do:\n- Explain concepts in simple terms or summarize long articles\n- Help write or edit emails, essays, resumes, or social posts\n- Debug code or walk through tech problems\n- Learn a topic step by step or study for a test\n- Plan something practical (a trip, a schedule, a meal plan, a project)\n\nWhat would you like to do first? If you’re not sure, tell me a bit about your interests and I’ll suggest something.'

## 开始我们的第一个项目

下面用 `fetch_website_contents` 抓网页正文，再交给模型做「运动鞋/站点」向摘要。


In [12]:
# ========== 试用抓取工具：Nike 首页正文 ==========

# URL 保持原样；返回值通常是清洗后的大段文本
ed = fetch_website_contents("https://www.nike.com/")
# 先打印看噪音（导航、页脚）有多少——后面 prompt 会要求忽略导航
print(ed)


Nike. Just Do It. Nike.com

Skip to main content
Accessibility at Nike
Find a Store
Help
Help
Order Status
Shipping & Delivery
Returns
Order Cancellation
Size Charts
Contact Us
Membership
Promotions & Discounts
Product Advice
Send Us Feedback
Join Us
Sign In
Men
New & Featured
New Arrivals
Best Sellers
Latest Drops
SNKRS Launch Calendar
Shop All Sale
Shoes
All Shoes
Basketball
Jordan
Lifestyle
Running
Sandals & Slides
Soccer
Training & Gym
Custom Shoes
Clothing
All Clothing
Hoodies & Sweatshirts
Jackets & Vests
Pants
Shorts
Swim
Tops & Graphic Tees
Accessories
All Accessories
Bags & Backpacks
Hats & Headwear
Socks
Women
New & Featured
New Arrivals
Best Sellers
SNKRS Launch Calendar
Shop All Sale
Shop by Color
Dark Neutrals
Crimson
Light Magenta
Orange Pulse
Steam Green
University Blue
Warm Neutrals
Shoes
All Shoes
Basketball
Jordan
Lifestyle
Running
Sandals & Slides
Soccer
Training & Gym
Custom Shoes
Clothing
All Clothing
Bras
Hoodies & Sweatshirts
Leggings
Matching Sets
Jackets & Vest

## 提示类型（Prompt Types）

像 GPT 这类聊天模型，通常期望两类指令：

- **System prompt（系统提示）**：任务是什么、语气/格式是什么
- **User prompt（用户提示）**：本轮具体输入（这里是网页正文 + 任务说明）


In [13]:
# ========== 定义 system prompt（角色 + Markdown 输出规则） ==========
# 可稍后实验：把最后一句改成 Spanish 等；但默认英文原文不要改译（会影响行为）

system_prompt = """
You are sports analyst that analyzes the contents of a website,
and provides a short, fact based, summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""


In [14]:
# ========== 定义 user prompt 前缀（后面会拼接网站正文） ==========

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements about sneakers, then summarize these too.
Suggest if there is any new sneaker worth investing for selling in future.

"""


## Messages（消息列表）

OpenAI API（以及很多兼容 API）期望这种结构：

```python
[
    {"role": "system", "content": "system message goes here"},
    {"role": "user", "content": "user message goes here"}
]
```

下一格先用一个很短的「运动鞋分析师」问答热身（还不用网页正文）。


In [15]:
# ========== 热身调用：system + user，模型 gpt-4.1-nano ==========

# 手写两条消息；content 英文保留
messages = [
    {"role": "system", "content": "You are a sports analyst "},
    {"role": "user", "content": "What is the  best selling sneaker?"}
]

# 发起聊天；注意这里用的是 gpt-4.1-nano（与后面摘要模型不同）
response = openai.chat.completions.create(model="gpt-4.1-nano", messages=messages)
# 取出回复文本
response.choices[0].message.content


'As of October 2023, the best-selling sneaker globally is the Nike Air Force 1. This iconic model has maintained its popularity for decades due to its timeless design, versatility, and cultural significance. Other popular contenders include sneakers like the Adidas Yeezy Boost series and the Nike Air Jordan line, but the Air Force 1 consistently holds a top spot in sales worldwide.'

## 用函数为 GPT-4.1-mini 组装 messages

把「system_prompt + user_prompt_prefix + 网页正文」封装成函数，避免每次手写列表。


In [16]:
# ========== messages_for：把网页正文塞进统一 messages 结构 ==========

def messages_for(website):
    # 返回与上一格相同形状的 list[dict]；user 内容 = 前缀 + 正文
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]


In [17]:
# ========== 试调用：用刚才抓到的 Nike 正文看 messages 长什么样 ==========

# 不会发 API；只是预览拼好的 prompt（可能很长）
messages_for(ed)


[{'role': 'system',
  'content': '\nYou are sports analyst that analyzes the contents of a website,\nand provides a short, fact based, summary, ignoring text that might be navigation related.\nRespond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.\n'},
 {'role': 'user',
  'content': '\nHere are the contents of a website.\nProvide a short summary of this website.\nIf it includes news or announcements about sneakers, then summarize these too.\nSuggest if there is any new sneaker worth investing for selling in future.\n\nNike. Just Do It. Nike.com\n\nSkip to main content\nAccessibility at Nike\nFind a Store\nHelp\nHelp\nOrder Status\nShipping & Delivery\nReturns\nOrder Cancellation\nSize Charts\nContact Us\nMembership\nPromotions & Discounts\nProduct Advice\nSend Us Feedback\nJoin Us\nSign In\nMen\nNew & Featured\nNew Arrivals\nBest Sellers\nLatest Drops\nSNKRS Launch Calendar\nShop All Sale\nShoes\nAll Shoes\nBasketball\nJordan\nLifestyle\nRunning

## 串起来：抓取 → messages → Chat Completions

OpenAI 的调用表面很简单；复杂点在 prompt 与输入清洗。


In [18]:
# ========== summarize(url)：抓取网页并调用 gpt-4.1-mini 做摘要 ==========

def summarize(url):
    # 1) 抓取正文
    website = fetch_website_contents(url)
    # 2) 调 Chat Completions；model id 保持 gpt-4.1-mini
    response = openai.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = messages_for(website)
    )
    # 3) 返回助手文本（Markdown 字符串）
    return response.choices[0].message.content


In [19]:
# ========== 对 Nike 跑一次摘要（返回字符串，尚未 Markdown 渲染） ==========
summarize("https://www.nike.com")


"The website is Nike's official online store, offering a wide range of products across categories such as men's, women's, kids', teens', and Jordan collections. It features new arrivals, best sellers, latest sneaker drops, and sale items. The site includes sections for various sports categories like basketball, running, soccer, and training, as well as lifestyle shoes, custom shoes, and clothing. Accessories, including bags, hats, and socks, are also available.\n\nRegarding sneaker news or announcements, the website highlights the SNKRS Launch Calendar, which likely details upcoming sneaker releases, though no specific new sneaker models or investment advice are directly mentioned in the content provided.\n\n**Investment Suggestion:**  \nThe SNKRS Launch Calendar represents the freshest and most limited sneaker releases from Nike and Jordan brands. These drops often include exclusive or collaboration models that tend to appreciate in value. While no specific sneaker is named here, moni

In [20]:
# ========== display_summary：摘要 + 笔记本 Markdown 展示 ==========

def display_summary(url):
    # 复用 summarize，再交给 IPython.display
    summary = summarize(url)
    display(Markdown(summary))


In [21]:
# ========== 渲染 Nike 摘要 ==========
display_summary("https://www.nike.com")


Nike.com is the official website for Nike, offering a wide range of products including footwear, clothing, and accessories for men, women, kids, and teens. The site features categories such as basketball, running, lifestyle, soccer, and training shoes, with dedicated sections for popular lines like Jordan and NikeSKIMS. It also provides options for custom shoes and a variety of apparel including hoodies, jackets, pants, shorts, and accessories like bags and hats. 

The website highlights new arrivals, best sellers, and the SNKRS launch calendar for upcoming sneaker releases. There is a focus on seasonal and color-themed collections, along with sales and promotions.

### Sneakers News & Announcements:
- The website includes a SNKRS Launch Calendar which details the latest sneaker drops.
- Featured collections like Jordan x Brasil Futebol and Nike x LEGO® suggest collaborations attracting collectors.
- New arrivals and best sellers are regularly updated, indicating fresh releases.

### Investment Suggestion:
- Sneakers from the Jordan line, especially limited editions like Jordan x Brasil Futebol, are often highly sought after and could be a good investment for future resale.
- Collaboration sneakers such as Nike x LEGO® may also hold or increase value due to their unique appeal and limited availability.

Overall, closely monitoring the SNKRS Launch Calendar and focusing on collaborative or special edition releases could present lucrative resale opportunities.

# 再试更多网站

注意：这种简单抓取**只对「服务器直接返回 HTML」的站点可靠**。

用 JavaScript 渲染的站点（很多 React 应用）可能几乎抓不到正文——社区贡献里有 Selenium 方案。  
被 CloudFront 等保护的站点也可能 403。

但很多新闻/企业站仍然可以。


In [18]:
# ========== 再试 CNN ==========
display_summary("https://cnn.com")


# CNN: The Grand Central Station of News (and Ads)

Welcome to CNN, where you'll find *everything* under the sun—US, world, politics, business, health, entertainment, and yes, even climate and weather, because what's news without a little apocalypse? They've got breaking news, updates on hot topics like the Ukraine-Russia and Israel-Hamas wars, and even the niche stuff like Epstein Files and Fear & Greed Index (because who doesn't love a little financial drama?).

But wait, there’s more! Your snarky assistant is particularly impressed by the intense ad feedback system, inviting you to report every imaginable ad misery, from “ad never loaded” to “audio on ad was too loud.” Nothing like a good ol’ interrupted scroll to keep your blood pressure up.

News and announcements? Nah, just the usual barrage of global crises, political chaos, celebrity scoops, and market jitters—served fresh daily. Plus, multimedia content, for those who like their news with a side of buffering and “video player was slow to load.”

In short: CNN is your deluxe all-you-can-eat buffet of news, seasoned generously with ads and a side dish of technical gripes. Bon appétit!

In [ ]:
# ========== 再试 Anthropic 官网 ==========
display_summary("https://anthropic.com")


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">商业应用</h2>
            <span style="color:#181;">你刚第一次体验了调用 Frontier Model 的云端 API。除了以后训练/微调自己的模型，课程很多阶段都会用 OpenAI 这类 API。<br/><br/>
            更具体：这里练的是<strong>摘要</strong>——经典 GenAI 用例。可迁移到新闻、财报、简历/求职信等。想想你的业务里哪里需要「先压缩信息再决策」，并试着做原型。</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">继续之前 — 现在自己试一下</h2>
            <span style="color:#900;">用下一格做你自己的小商业例子。仍可围绕摘要：例如根据邮件正文建议短主题行，并列出待办——这很像商务邮箱助手的雏形。</span>
        </td>
    </tr>
</table>


In [ ]:
# ========== 练习扩展：邮件摘要 + 建议主题行 + Action Items ==========
# Step 1: Create your prompts — system/user 字符串保持英文原样（改译会改行为）

system_prompt = "you are an admin assistant trying to summarize the contents of emails coming to the inbox. Summarize the email in bullet points, suggest a short subject line for the email. suggest any action items for the user"
user_prompt = """
    Hi Vidit,

Very nice connecting with you, thought you would be an excellent fit for this completely remote position!

In this role, you will work directly with clients to understand their needs, walk them through benefit options, and ensure they receive exceptional  support from start to finish. This position blends client engagement, account management, and benefits guidance with a forward-thinking,  service-driven approach.

Responsibilities

Act as the main point of contact for client regarding their employee benefits, including health, dental, vision, life, disability and retirement plans.
Help clients understand their benefit options, explain costs and  features, and recommend solutions tailored to their organizational goals.
Lead individual client onboarding, ensuring smooth plan setup and seamless enrollment.
Daily Duties 

Host virtual meetings to review benefit plans, renewals, updates, and enhancements.
Build strong, lasting relationships with clients and identify opportunities to expand the services they use.
Work cross-functionally with internal teams to ensure accuracy, timely responses, and a smooth client experience.
Stay informed on industry trends and regulatory changes to provide engaging, knowledgeable support.
Qualifications 

Excellent communication and presentation skills—comfort with breaking down complex topics for clients.
Experience using CRM or benefits administration tools (e.g., Salesforce, Ease, Gusto).
Strong organizational skills, attention to detail, and the ability to manage multiple client needs at once.
Self-starter mindset with the ability to excel in a fully remote environment.
Desired Skills

Excellent communication skills
Use company CRMS
Client focused mindset
Self motivated & comfortable working from home
Check out the opportunities below to apply:

Benefits Coordinator (Fully Remote)
Looking for a new opportunity? Click here to see all our open roles!

Want to change how you receive these emails? Update your preferences here
"""

# Step 2: 组装 messages（system 定角色，user 放邮件正文）
messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ] # fill this in

# Step 3: Call OpenAI — model id 保持 gpt-4.1-nano
response = openai.chat.completions.create(model="gpt-4.1-nano", messages=messages)
response.choices[0].message.content

# Step 4: print the result — 再打印一次便于阅读
print(response.choices[0].message.content)


**Email Summary:**
- Connecting with Vidit about a fully remote benefits coordinator role.
- Role involves client interaction around employee benefits, explaining options, and ensuring support from onboarding to ongoing service.
- Responsibilities include managing client benefits (health, dental, vision, life, disability, retirement), hosting virtual meetings, onboarding clients, building relationships, and collaborating internally.
- Requires strong communication, organizational skills, experience with CRM tools, and ability to work independently.
- Encouragement to explore application opportunities for the role and other open positions.

**Suggested Subject Line:**  
Remote Benefits Coordinator Opportunity

**Action Items:**  
- Review the details of the benefits coordinator role.
- Consider applying if interested.
- Update email preferences if you want to modify communication settings.


## 额外练习：更强的网页抓取

若你试 `display_summary("https://openai.com")` 可能失败——该站大量依赖 JavaScript。  
常见对策：Selenium / Playwright 在真实浏览器里渲染后再取文本。社区贡献文件夹里有同学的 Selenium 示例。有经验的话，可以改进 Website/scraper 封装。


# 分享你的代码

欢迎把改进（含 Selenium）放到社区贡献目录并发 PR，方便他人学习。

PR 总览：https://edwarddonner.com/pr  

提交前自检：

1. PR 尽量只含 community-contributions 相关改动（除非另有约定）
2. 笔记本输出保持清晰
3. 总量别太大（例如别堆几千行无关文件）
4. 不要提交密钥、冗长测试垃圾、或无意义的 LLM 产物

详细步骤示例：https://chatgpt.com/share/6873c22b-2a1c-8012-bc9a-debdcf7c835b  
指南文件夹里也有 git/PR 说明（Guide 3）。


In [ ]:
# （空单元格保留）可在此继续实验：换 URL、改 system_prompt、或接 Selenium
